# 🎯 Zalo AI Challenge 2026: DataMate — Chạy Suy Luận (Inference) với Mô Hình SFT LoRA Qwen 7B

> **Mục tiêu:** Sử dụng trực tiếp mô hình **Qwen2.5-Coder-7B-Instruct** kết hợp với **LoRA Adapter đã huấn luyện** (`result/sft_adapter_qwen7b/final_adapter`) để chạy suy luận (predict) mà **KHÔNG CẦN TRAIN LẠI**.
>
> **Kết quả đầu ra:** Xuất file dự đoán `result/submission.jsonl` và bộ logs chi tiết theo chuẩn thể lệ Zalo AI Challenge.

## Bước 1: Thiết lập môi trường & Kết nối Google Drive
Tự động kết nối Drive (có cơ chế bypass không hỏi popup lại nếu đã kết nối) và đồng bộ dự án sang SSD Colab.

In [ ]:
import os, shutil

# Tự động bypass: Nếu Drive đã mount thì không gọi lại
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('✅ Google Drive đã được kết nối từ trước!')

gdrive_project = '/content/drive/MyDrive/Colab Notebooks/ZALO-AI-CHALLENGE-2026'
local_work = '/content/datamate'

assert os.path.exists(gdrive_project), f'Không tìm thấy thư mục dự án trên Drive: {gdrive_project}'

os.makedirs(local_work, exist_ok=True)
!rsync -a --exclude='.git' --exclude='tmp' --exclude='wandb' --exclude='__pycache__' --exclude='.ipynb_checkpoints' "$gdrive_project/" "$local_work/"

os.chdir(local_work)
print(f'✅ Workspace làm việc: {os.getcwd()}')

## Bước 2: Cài đặt Dependencies
Cài đặt các thư viện cần thiết để chạy inference.

In [ ]:
# Cài đặt dependencies (thêm bitsandbytes để chạy 4-bit siêu nhẹ trên T4)
!pip install -q transformers peft accelerate bitsandbytes python-calamine pyyaml tqdm
!pip uninstall -y torchao -q


## Bước 3: Kiểm tra LoRA Adapter đã huấn luyện
Xác nhận các file trọng số LoRA (`adapter_model.safetensors`, `adapter_config.json`) đã có sẵn trong workspace.

In [ ]:
adapter_dir = 'result/sft_adapter_qwen7b/final_adapter'
assert os.path.exists(adapter_dir), f'Chưa tìm thấy adapter tại {adapter_dir}! Kiểm tra lại Drive.'
print(f'✅ Đã tìm thấy LoRA Adapter tại: {adapter_dir}')
!ls -la "$adapter_dir"

## Bước 4: Chạy Suy Luận (Inference)
Chọn 1 trong 2 chế độ suy luận bên dưới:

In [ ]:
import os, torch

# ====================================================================
# TỰ ĐỘNG CHỌN CẤU HÌNH THEO GPU (T4 TIẾT KIỆM vs A100/L4)
# ====================================================================
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
is_t4 = 'T4' in gpu_name
# Trên T4: dùng quantization 4-bit (chỉ tốn ~4.8GB VRAM) để không bao giờ bị OOM
# Trên A100/L4: dùng bfloat16 gốc
config_file = 'configs/sft_trained_7b_t4.yaml' if is_t4 else 'configs/sft_trained_7b.yaml'

print(f'🖥️ GPU phát hiện: {gpu_name}')
print(f'⚙️ Cấu hình suy luận: {config_file} (T4 4-bit mode: {is_t4})')

# Chọn dữ liệu cần chạy (Mặc định: Public Test)
TARGET_DATA = 'data/public_test'
OUTPUT_SUBMISSION = 'result/submission_public_test.jsonl'
LOGS_DIR = 'result/logs_public_test'

!python scripts/run_inference.py \
    --config $config_file \
    --data-dir $TARGET_DATA \
    --output $OUTPUT_SUBMISSION \
    --logs-dir $LOGS_DIR


## Bước 5: Kiểm tra tính hợp lệ của file kết quả
Kiểm tra định dạng, số dòng và phân bố các kiểu câu trả lời.

In [ ]:
import json
from collections import Counter

assert os.path.exists(OUTPUT_SUBMISSION), f'Chưa có file {OUTPUT_SUBMISSION}'

with open(OUTPUT_SUBMISSION, 'r', encoding='utf-8') as f:
    lines = [json.loads(line) for line in f if line.strip()]

print(f'✅ Tổng số câu đã giải: {len(lines)}')
type_counts = Counter(item.get('type') for item in lines)
print('📊 Phân bố kiểu đáp án:', dict(type_counts))
print('\n5 mẫu kết quả đầu tiên:')
for item in lines[:5]:
    print(' ', item)

## Bước 6: Lưu kết quả về Google Drive
Tự động copy file `submission.jsonl` và thư mục logs về Google Drive vĩnh viễn.

In [ ]:
drive_result_dir = f'{gdrive_project}/result'
!cp $OUTPUT_SUBMISSION "$drive_result_dir/"
!cp -r $LOGS_DIR "$drive_result_dir/"
print(f'✅ Đã lưu kết quả về Drive thành công tại: {drive_result_dir}')